In [1]:
from pathlib import Path
import sys

import joblib
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.train import build_svm_pipeline

print("Project root:", PROJECT_ROOT)

Project root: /home/jadkandah/progressSoft_internship/phase-1-machine-learning-nlp/assignment


In [2]:
TRAIN_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "train_clean.csv"
)

train_df = pd.read_csv(TRAIN_PATH)

print("Training shape:", train_df.shape)
display(train_df.head())

Training shape: (69384, 4)


,tweet_id,entity,sentiment,text
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,2401,Borderlands,Positive,i am coming to the borders and i will kill you...
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...


In [3]:
X_train = train_df["text"]
y_train = train_df["sentiment"]

print("Training samples:", len(X_train))
print("\nLabel distribution:")
print(y_train.value_counts())

Training samples: 69384

Label distribution:
sentiment
Negative      21157
Positive      19068
Neutral       16978
Irrelevant    12181
Name: count, dtype: int64


In [5]:
BEST_SVM_PARAMS = {
    "classifier__C": 2.0,
    "classifier__class_weight": None,
}
print(BEST_SVM_PARAMS)

{'classifier__C': 2.0, 'classifier__class_weight': None}


In [6]:
final_model = build_svm_pipeline()

final_model.set_params(**BEST_SVM_PARAMS)

final_model.fit(X_train, y_train)

print("Final model trained successfully.")

Final model trained successfully.


In [7]:
MODELS_DIR = PROJECT_ROOT / "models"

MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MODEL_PATH = MODELS_DIR / "sentiment_pipeline.joblib"

print("Model path:", MODEL_PATH)

Model path: /home/jadkandah/progressSoft_internship/phase-1-machine-learning-nlp/assignment/models/sentiment_pipeline.joblib


In [8]:
joblib.dump(
    final_model,
    MODEL_PATH,
)

print(f"Model saved to: {MODEL_PATH}")

Model saved to: /home/jadkandah/progressSoft_internship/phase-1-machine-learning-nlp/assignment/models/sentiment_pipeline.joblib


In [9]:
loaded_model = joblib.load(MODEL_PATH)

print("Model loaded successfully.")
print(loaded_model)

Model loaded successfully.
Pipeline(steps=[('tfidf',
                 TfidfVectorizer(lowercase=False, max_df=0.98,
                                 max_features=100000, min_df=2,
                                 ngram_range=(1, 2),
                                 preprocessor=<function normalize_text at 0x72e787b7ff60>,
                                 sublinear_tf=True, token_pattern=None,
                                 tokenizer=<function tokenize_text at 0x72e786e4de40>)),
                ('classifier', LinearSVC(C=2.0, random_state=42))])


In [11]:
test_texts = [
    "I absolutely love this new update!",
    "This game is terrible and completely broken.",
    "The maintenance update starts tomorrow.",
    "I cooked pasta while watching television.",
]

predictions = loaded_model.predict(test_texts)

test_results = pd.DataFrame(
    {
        "text": test_texts,
        "predicted_sentiment": predictions,
    }
)

test_results

,text,predicted_sentiment
0,I absolutely love this new update!,Positive
1,This game is terrible and completely broken.,Negative
2,The maintenance update starts tomorrow.,Neutral
3,I cooked pasta while watching television.,Positive


In [12]:
original_predictions = final_model.predict(test_texts)
loaded_predictions = loaded_model.predict(test_texts)

assert (
    original_predictions == loaded_predictions
).all()

print("Saved and reloaded model predictions match.")

Saved and reloaded model predictions match.
